In [1]:
import pandas as pd
import numpy as np
import re, os
from pathlib import Path
from itertools import product

In [2]:
# ============================================================
# 7A. Load transcript-mapped dataset
# ============================================================

INPUT_PATH = Path(
    r"C:\Users\oozgen\Desktop\phd\dataset\training\05_transcript_coordinate_mapping\SOD1_process_v3_transcript_mapping.csv"
)

df = pd.read_csv(INPUT_PATH)
df.columns = df.columns.str.strip()

print("Loaded:", INPUT_PATH)
print("Shape:", df.shape)

df.head()

Loaded: C:\Users\oozgen\Desktop\phd\dataset\training\05_transcript_coordinate_mapping\SOD1_process_v3_transcript_mapping.csv
Shape: (1026, 49)


,ISIS,Sequence,Modification,Location,Chemical_Pattern,Linkage,Linkage_Location,Smiles,Inhibition(%),seq_length,...,transcript_description,transcript_length,n_transcript_hits,target_start_0based,target_end_0based_exclusive,target_start_1based,target_end_1based_inclusive,matched_transcript_sequence,mapping_orientation,transcript_mapping_attempted
0,150478,ATTGAAACAGACATTTTAAC,MOE/5-methylcytosines/deoxy,0?1?2?3?4?15?16?17?18?19/C/else,MMMMMddddddddddMMMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](O)[C@@H](C[O]P(=O)([S-])O[C@...,0,20,...,NM_000454.5 Homo sapiens superoxide dismutase ...,895,1,739.0,759.0,740.0,759.0,GTTAAAATGTCTGTTTCAAT,ASO_reverse_complement_matches_transcript,True
1,150494,TCATAATAAGTGCCATACAG,MOE/5-methylcytosines/deoxy,0?1?2?3?4?15?16?17?18?19/C/else,MMMMMddddddddddMMMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,0,20,...,NM_000454.5 Homo sapiens superoxide dismutase ...,895,1,844.0,864.0,845.0,864.0,CTGTATGGCACTTATTATGA,ASO_reverse_complement_matches_transcript,True
2,489531,AAATCAGTTTCTCACTACAG,MOE/5-methylcytosines/deoxy,0?1?2?3?4?15?16?17?18?19/C/else,MMMMMddddddddddMMMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,0,20,...,NM_000454.5 Homo sapiens superoxide dismutase ...,895,1,680.0,700.0,681.0,700.0,CTGTAGTGAGAAACTGATTT,ASO_reverse_complement_matches_transcript,True
3,590442,CGCTGCAGGAGACTACG,MOE/cEt/5-methylcytosines/deoxy,0?1?2?14?15?16/3?4?12?13/C/else,MMMCCdddddddCCMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,0,17,...,NM_000454.5 Homo sapiens superoxide dismutase ...,895,1,4.0,21.0,5.0,21.0,CGTAGTCTCCTGCAGCG,ASO_reverse_complement_matches_transcript,True
4,590455,CTTTCCTTCTGCTCGAA,MOE/cEt/5-methylcytosines/deoxy,0?1?2?14?15?16/3?4?12?13/C/else,MMMCCdddddddCCMMM,phosphorothioate,else,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,0,17,...,NM_000454.5 Homo sapiens superoxide dismutase ...,895,1,137.0,154.0,138.0,154.0,TTCGAGCAGAAGGAAAG,ASO_reverse_complement_matches_transcript,True


In [4]:
# ============================================================
# Output folders
# ============================================================

TRAINING_DIR = Path(r"C:\Users\oozgen\Desktop\phd\dataset\training")

STEP7_DIR = TRAINING_DIR / "06_biophysical_features"
STEP8_DIR = TRAINING_DIR / "07_model_ready_dataset"

STEP7_DIR.mkdir(parents=True, exist_ok=True)
STEP8_DIR.mkdir(parents=True, exist_ok=True)

print("Step 7 output folder:")
print(STEP7_DIR)

print("\nStep 8 output folder:")
print(STEP8_DIR)

Step 7 output folder:
C:\Users\oozgen\Desktop\phd\dataset\training\06_biophysical_features

Step 8 output folder:
C:\Users\oozgen\Desktop\phd\dataset\training\07_model_ready_dataset


In [5]:
# ============================================================
# Output file paths
# ============================================================

STEP7A_OUT = STEP7_DIR / "SOD1_process_v4a_sequence_features.csv"
STEP7B_OUT = STEP7_DIR / "SOD1_process_v4b_gap_architecture_features.csv"
STEP7C_OUT = STEP7_DIR / "SOD1_process_v4c_gap_3mer_features.csv"
STEP7D_OUT = STEP7_DIR / "SOD1_process_v4d_transcript_coordinate_features.csv"
STEP7E_OUT = STEP7_DIR / "SOD1_process_v4e_basic_tm_features.csv"
STEP7F_OUT = STEP7_DIR / "SOD1_process_v4f_rdkit_features.csv"

FINAL_OUT = STEP8_DIR / "SOD1_model_ready_v1.csv"
FINAL_DATA_DICTIONARY_OUT = STEP8_DIR / "SOD1_model_ready_v1_data_dictionary.csv"
FINAL_VALIDATION_OUT = STEP8_DIR / "SOD1_model_ready_v1_validation_summary.csv"

In [6]:
# 7B. Basic sequence features
# gc_fraction
# base counts
# base fractions
# longest homopolymer
# homopolymer flag
# maximum GC stretch

In [7]:
# ============================================================
# 7B. Basic sequence features
# ============================================================

required_cols = ["ISIS", "Sequence", "seq_length"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for sequence features: {missing_cols}")

df["Sequence"] = df["Sequence"].astype(str).str.upper().str.strip()
df["seq_length"] = pd.to_numeric(df["seq_length"], errors="coerce").astype(int)

def gc_fraction(seq):
    seq = str(seq).upper()
    return (seq.count("G") + seq.count("C")) / len(seq)

def count_base(seq, base):
    seq = str(seq).upper()
    return seq.count(base)

def longest_homopolymer(seq):
    seq = str(seq).upper()

    if len(seq) == 0:
        return 0

    max_run = 1
    current = 1

    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 1

    return max_run

def max_gc_stretch(seq):
    seq = str(seq).upper()

    max_run = 0
    current = 0

    for base in seq:
        if base in {"G", "C"}:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0

    return max_run

# Drop old columns if rerunning
sequence_feature_cols = [
    "gc_fraction",
    "n_A",
    "n_C",
    "n_G",
    "n_T",
    "frac_A",
    "frac_C",
    "frac_G",
    "frac_T",
    "longest_homopolymer",
    "has_homopolymer_4plus",
    "has_homopolymer_5plus",
    "max_gc_stretch",
]

df = df.drop(columns=[c for c in sequence_feature_cols if c in df.columns], errors="ignore")

df["gc_fraction"] = df["Sequence"].apply(gc_fraction)

df["n_A"] = df["Sequence"].apply(lambda x: count_base(x, "A"))
df["n_C"] = df["Sequence"].apply(lambda x: count_base(x, "C"))
df["n_G"] = df["Sequence"].apply(lambda x: count_base(x, "G"))
df["n_T"] = df["Sequence"].apply(lambda x: count_base(x, "T"))

df["frac_A"] = df["n_A"] / df["seq_length"]
df["frac_C"] = df["n_C"] / df["seq_length"]
df["frac_G"] = df["n_G"] / df["seq_length"]
df["frac_T"] = df["n_T"] / df["seq_length"]

df["longest_homopolymer"] = df["Sequence"].apply(longest_homopolymer)
df["has_homopolymer_4plus"] = (df["longest_homopolymer"] >= 4).astype(int)
df["has_homopolymer_5plus"] = (df["longest_homopolymer"] >= 5).astype(int)

df["max_gc_stretch"] = df["Sequence"].apply(max_gc_stretch)

# Validation: base counts should sum to sequence length
df["base_count_sum_valid"] = (
    df["n_A"] + df["n_C"] + df["n_G"] + df["n_T"]
) == df["seq_length"]

print("Base count validation:")
print(df["base_count_sum_valid"].value_counts(dropna=False))

if not df["base_count_sum_valid"].all():
    bad_rows = df.loc[
        ~df["base_count_sum_valid"],
        ["ISIS", "Sequence", "seq_length", "n_A", "n_C", "n_G", "n_T"]
    ]
    display(bad_rows)
    raise ValueError("Base count validation failed.")

df.to_csv(STEP7A_OUT, index=False)

print("Saved:")
print(STEP7A_OUT)

df[
    [
        "ISIS",
        "Sequence",
        "seq_length",
        "gc_fraction",
        "n_A",
        "n_C",
        "n_G",
        "n_T",
        "longest_homopolymer",
        "max_gc_stretch",
    ]
].head()

Base count validation:
base_count_sum_valid
True    1026
Name: count, dtype: int64
Saved:
C:\Users\oozgen\Desktop\phd\dataset\training\06_biophysical_features\SOD1_process_v4a_sequence_features.csv


,ISIS,Sequence,seq_length,gc_fraction,n_A,n_C,n_G,n_T,longest_homopolymer,max_gc_stretch
0,150478,ATTGAAACAGACATTTTAAC,20,0.250000,9,3,2,6,4,1
1,150494,TCATAATAAGTGCCATACAG,20,0.350000,8,4,3,5,2,3
2,489531,AAATCAGTTTCTCACTACAG,20,0.350000,7,5,2,6,3,1
3,590442,CGCTGCAGGAGACTACG,17,0.647059,4,5,6,2,2,3
4,590455,CTTTCCTTCTGCTCGAA,17,0.470588,2,6,2,7,3,2


In [8]:
# 7C. Gap architecture features
# gap_start_0based
# gap_end_0based_exclusive
# gap_length
# gap_sequence
# flank5_length
# flank3_length
# flank_asymmetry_5minus3
# longest_DNA_gap

In [9]:
# ============================================================
# 7C. Gap architecture features
# ============================================================

df = pd.read_csv(STEP7A_OUT)
df.columns = df.columns.str.strip()

required_cols = ["ISIS", "Sequence", "Chemical_Pattern", "seq_length"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for gap features: {missing_cols}")

df["Sequence"] = df["Sequence"].astype(str).str.upper().str.strip()
df["Chemical_Pattern"] = df["Chemical_Pattern"].astype(str).str.strip()

def longest_contiguous_dna_gap(pattern):
    pattern = str(pattern)

    max_run = 0
    current = 0

    for symbol in pattern:
        if symbol == "d":
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0

    return max_run

def dna_gap_bounds(pattern):
    """
    Returns the start/end of the longest contiguous DNA gap.
    Coordinates are 0-based and end-exclusive.
    """
    pattern = str(pattern)

    best_start = pd.NA
    best_end = pd.NA
    best_len = 0

    current_start = None
    current_len = 0

    for i, symbol in enumerate(pattern):
        if symbol == "d":
            if current_start is None:
                current_start = i

            current_len += 1

            if current_len > best_len:
                best_len = current_len
                best_start = current_start
                best_end = i + 1
        else:
            current_start = None
            current_len = 0

    return best_start, best_end, best_len

def get_gap_sequence(seq, pattern):
    start, end, gap_len = dna_gap_bounds(pattern)

    if pd.isna(start) or pd.isna(end):
        return ""

    return str(seq)[int(start):int(end)]

def flank_lengths(pattern):
    start, end, gap_len = dna_gap_bounds(pattern)

    if pd.isna(start) or pd.isna(end):
        return 0, 0

    pattern = str(pattern)

    flank5 = int(start)
    flank3 = len(pattern) - int(end)

    return flank5, flank3

gap_feature_cols = [
    "longest_DNA_gap",
    "gap_start_0based",
    "gap_end_0based_exclusive",
    "gap_length",
    "gap_sequence",
    "flank5_length",
    "flank3_length",
    "flank_asymmetry_5minus3",
    "gap_length_valid",
]

df = df.drop(columns=[c for c in gap_feature_cols if c in df.columns], errors="ignore")

df["longest_DNA_gap"] = df["Chemical_Pattern"].apply(longest_contiguous_dna_gap)

gap_info = df["Chemical_Pattern"].apply(dna_gap_bounds).apply(pd.Series)
gap_info.columns = [
    "gap_start_0based",
    "gap_end_0based_exclusive",
    "gap_length",
]

df = pd.concat([df.reset_index(drop=True), gap_info.reset_index(drop=True)], axis=1)

df["gap_sequence"] = df.apply(
    lambda row: get_gap_sequence(row["Sequence"], row["Chemical_Pattern"]),
    axis=1,
)

flank_info = df["Chemical_Pattern"].apply(flank_lengths).apply(pd.Series)
flank_info.columns = ["flank5_length", "flank3_length"]

df = pd.concat([df.reset_index(drop=True), flank_info.reset_index(drop=True)], axis=1)

df["flank_asymmetry_5minus3"] = df["flank5_length"] - df["flank3_length"]

# Validate gap length
df["gap_length_valid"] = df["gap_sequence"].str.len() == df["gap_length"]

print("Gap length validation:")
print(df["gap_length_valid"].value_counts(dropna=False))

if not df["gap_length_valid"].all():
    bad_rows = df.loc[
        ~df["gap_length_valid"],
        ["ISIS", "Sequence", "Chemical_Pattern", "gap_sequence", "gap_length"]
    ]
    display(bad_rows)
    raise ValueError("Gap length validation failed.")

df.to_csv(STEP7B_OUT, index=False)

print("Saved:")
print(STEP7B_OUT)

df[
    [
        "ISIS",
        "Sequence",
        "Chemical_Pattern",
        "gap_sequence",
        "gap_length",
        "flank5_length",
        "flank3_length",
        "flank_asymmetry_5minus3",
    ]
].head()

Gap length validation:
gap_length_valid
True    1026
Name: count, dtype: int64
Saved:
C:\Users\oozgen\Desktop\phd\dataset\training\06_biophysical_features\SOD1_process_v4b_gap_architecture_features.csv


,ISIS,Sequence,Chemical_Pattern,gap_sequence,gap_length,flank5_length,flank3_length,flank_asymmetry_5minus3
0,150478,ATTGAAACAGACATTTTAAC,MMMMMddddddddddMMMMM,AACAGACATT,10,5,5,0
1,150494,TCATAATAAGTGCCATACAG,MMMMMddddddddddMMMMM,ATAAGTGCCA,10,5,5,0
2,489531,AAATCAGTTTCTCACTACAG,MMMMMddddddddddMMMMM,AGTTTCTCAC,10,5,5,0
3,590442,CGCTGCAGGAGACTACG,MMMCCdddddddCCMMM,CAGGAGA,7,5,5,0
4,590455,CTTTCCTTCTGCTCGAA,MMMCCdddddddCCMMM,CTTCTGC,7,5,5,0


In [10]:
# 7D. Gap 3-mer features

In [11]:
# ============================================================
# 7D. Gap 3-mer features
# ============================================================

df = pd.read_csv(STEP7B_OUT)
df.columns = df.columns.str.strip()

required_cols = ["ISIS", "gap_sequence", "gap_length"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for gap 3-mer features: {missing_cols}")

kmers_3 = ["".join(p) for p in product("ACGT", repeat=3)]

def count_gap_3mers(gap_seq):
    gap_seq = str(gap_seq).upper()

    counts = {f"gap_3mer_{kmer}": 0 for kmer in kmers_3}

    if len(gap_seq) < 3:
        return counts

    for i in range(len(gap_seq) - 3 + 1):
        kmer = gap_seq[i:i + 3]
        key = f"gap_3mer_{kmer}"

        if key in counts:
            counts[key] += 1

    return counts

# Drop old gap 3-mer columns if rerunning
old_kmer_cols = [c for c in df.columns if c.startswith("gap_3mer_")]
df = df.drop(columns=old_kmer_cols, errors="ignore")

gap_kmer_df = pd.DataFrame(
    df["gap_sequence"].apply(count_gap_3mers).tolist()
)

df = pd.concat([df.reset_index(drop=True), gap_kmer_df.reset_index(drop=True)], axis=1)

# Validation:
# For a gap of length L, total 3-mer counts should be max(L - 2, 0)
df["gap_3mer_total_count"] = gap_kmer_df.sum(axis=1)
df["gap_3mer_expected_count"] = df["gap_length"].apply(lambda x: max(int(x) - 2, 0))
df["gap_3mer_count_valid"] = df["gap_3mer_total_count"] == df["gap_3mer_expected_count"]

print("Gap 3-mer count validation:")
print(df["gap_3mer_count_valid"].value_counts(dropna=False))

if not df["gap_3mer_count_valid"].all():
    bad_rows = df.loc[
        ~df["gap_3mer_count_valid"],
        [
            "ISIS",
            "gap_sequence",
            "gap_length",
            "gap_3mer_total_count",
            "gap_3mer_expected_count",
        ]
    ]
    display(bad_rows)
    raise ValueError("Gap 3-mer count validation failed.")

df.to_csv(STEP7C_OUT, index=False)

print("Saved:")
print(STEP7C_OUT)
print("Added gap 3-mer columns:", gap_kmer_df.shape[1])

df[
    [
        "ISIS",
        "gap_sequence",
        "gap_length",
        "gap_3mer_total_count",
        "gap_3mer_expected_count",
        "gap_3mer_count_valid",
    ]
].head()

Gap 3-mer count validation:
gap_3mer_count_valid
True    1026
Name: count, dtype: int64
Saved:
C:\Users\oozgen\Desktop\phd\dataset\training\06_biophysical_features\SOD1_process_v4c_gap_3mer_features.csv
Added gap 3-mer columns: 64


,ISIS,gap_sequence,gap_length,gap_3mer_total_count,gap_3mer_expected_count,gap_3mer_count_valid
0,150478,AACAGACATT,10,8,8,True
1,150494,ATAAGTGCCA,10,8,8,True
2,489531,AGTTTCTCAC,10,8,8,True
3,590442,CAGGAGA,7,5,5,True
4,590455,CTTCTGC,7,5,5,True


In [12]:
# 7E. Transcript-coordinate-derived features

In [14]:
# ============================================================
# 7E. Transcript-coordinate-derived features — optimized version
# ============================================================

df = pd.read_csv(STEP7C_OUT)
df.columns = df.columns.str.strip()

# Defragment after many previous column additions, especially gap_3mer_* columns
df = df.copy()

coord_feature_cols = [
    "target_midpoint_1based",
    "target_relative_start",
    "target_relative_end",
    "target_relative_midpoint",
    "target_has_exact_refseq_match",
    "target_mapping_unique",
    "target_mapping_multiple",
    "target_context_5nt_each_side",
    "target_context_10nt_each_side",
    "target_context_20nt_each_side",
]

# Drop old coordinate-derived columns if rerunning
df = df.drop(columns=[c for c in coord_feature_cols if c in df.columns], errors="ignore")
df = df.copy()

required_mapping_cols = [
    "n_transcript_hits",
    "target_start_1based",
    "target_end_1based_inclusive",
    "transcript_length",
    "mapping_orientation",
]

missing_mapping_cols = sorted(set(required_mapping_cols) - set(df.columns))

if len(missing_mapping_cols) > 0:
    print("Some transcript mapping columns are missing.")
    print("Missing:", missing_mapping_cols)
    print("Coordinate-derived features will be set to NA / False.")

    coord_features = pd.DataFrame({
        "target_midpoint_1based": pd.Series([pd.NA] * len(df), index=df.index),
        "target_relative_start": pd.Series([pd.NA] * len(df), index=df.index),
        "target_relative_end": pd.Series([pd.NA] * len(df), index=df.index),
        "target_relative_midpoint": pd.Series([pd.NA] * len(df), index=df.index),
        "target_has_exact_refseq_match": pd.Series([False] * len(df), index=df.index),
        "target_mapping_unique": pd.Series([False] * len(df), index=df.index),
        "target_mapping_multiple": pd.Series([False] * len(df), index=df.index),
    })

else:
    # Convert mapping columns safely to numeric
    n_hits = pd.to_numeric(df["n_transcript_hits"], errors="coerce")
    target_start = pd.to_numeric(df["target_start_1based"], errors="coerce")
    target_end = pd.to_numeric(df["target_end_1based_inclusive"], errors="coerce")
    transcript_length = pd.to_numeric(df["transcript_length"], errors="coerce")

    target_midpoint = (target_start + target_end) / 2

    coord_features = pd.DataFrame({
        "target_midpoint_1based": target_midpoint,
        "target_relative_start": target_start / transcript_length,
        "target_relative_end": target_end / transcript_length,
        "target_relative_midpoint": target_midpoint / transcript_length,
        "target_has_exact_refseq_match": n_hits.fillna(0).astype(int) > 0,
        "target_mapping_unique": n_hits.fillna(0).astype(int) == 1,
        "target_mapping_multiple": n_hits.fillna(0).astype(int) > 1,
    }, index=df.index)

    print("Mapping status:")
    print(df["mapping_orientation"].value_counts(dropna=False))

    print("\nExact mapping:")
    print(coord_features["target_has_exact_refseq_match"].value_counts(dropna=False))

    print("\nUnique mapping:")
    print(coord_features["target_mapping_unique"].value_counts(dropna=False))

# Add all coordinate-derived features at once
df = pd.concat([df, coord_features], axis=1)

# Defragment after concat
df = df.copy()

Mapping status:
mapping_orientation
ASO_reverse_complement_matches_transcript    982
no_exact_match                                44
Name: count, dtype: int64

Exact mapping:
target_has_exact_refseq_match
True     982
False     44
Name: count, dtype: int64

Unique mapping:
target_mapping_unique
True     982
False     44
Name: count, dtype: int64


In [16]:
# ============================================================
# Optional: add transcript context windows from fetched FASTA
# Optimized version
# ============================================================

STEP6_DIR = TRAINING_DIR / "05_transcript_coordinate_mapping"
FETCHED_FASTA = Path(r"C:\Users\oozgen\Desktop\phd\dataset\training\05_transcript_coordinate_mapping\SOD1_refseq_transcript_fetched_from_NCBI.fasta")

def extract_context_from_transcript(transcript_seq, start_1based, end_1based, flank):
    """
    Extracts target region plus flanking bases from transcript sequence.
    Coordinates are 1-based inclusive.
    """
    if pd.isna(start_1based) or pd.isna(end_1based):
        return pd.NA

    start_0based = int(start_1based) - 1
    end_exclusive = int(end_1based)

    context_start = max(0, start_0based - flank)
    context_end = min(len(transcript_seq), end_exclusive + flank)

    return transcript_seq[context_start:context_end]


if FETCHED_FASTA.exists():
    try:
        from Bio import SeqIO

        record = next(SeqIO.parse(str(FETCHED_FASTA), "fasta"))
        transcript_seq = str(record.seq).upper().replace("U", "T")

        context_features = pd.DataFrame({
            "target_context_5nt_each_side": df.apply(
                lambda row: extract_context_from_transcript(
                    transcript_seq,
                    row.get("target_start_1based", pd.NA),
                    row.get("target_end_1based_inclusive", pd.NA),
                    flank=5,
                ),
                axis=1,
            ),
            "target_context_10nt_each_side": df.apply(
                lambda row: extract_context_from_transcript(
                    transcript_seq,
                    row.get("target_start_1based", pd.NA),
                    row.get("target_end_1based_inclusive", pd.NA),
                    flank=10,
                ),
                axis=1,
            ),
            "target_context_20nt_each_side": df.apply(
                lambda row: extract_context_from_transcript(
                    transcript_seq,
                    row.get("target_start_1based", pd.NA),
                    row.get("target_end_1based_inclusive", pd.NA),
                    flank=20,
                ),
                axis=1,
            ),
        }, index=df.index)

        print("Added transcript context windows from:")
        print(FETCHED_FASTA)

    except ImportError:
        print("Biopython not installed. Transcript context extraction skipped.")

        context_features = pd.DataFrame({
            "target_context_5nt_each_side": pd.Series([pd.NA] * len(df), index=df.index),
            "target_context_10nt_each_side": pd.Series([pd.NA] * len(df), index=df.index),
            "target_context_20nt_each_side": pd.Series([pd.NA] * len(df), index=df.index),
        })

else:
    context_features = pd.DataFrame({
        "target_context_5nt_each_side": pd.Series([pd.NA] * len(df), index=df.index),
        "target_context_10nt_each_side": pd.Series([pd.NA] * len(df), index=df.index),
        "target_context_20nt_each_side": pd.Series([pd.NA] * len(df), index=df.index),
    })

    print("Fetched FASTA file not found. Transcript context windows set to NA.")

# Add context features all at once
df = pd.concat([df, context_features], axis=1)

# Defragment
df = df.copy()

Added transcript context windows from:
C:\Users\oozgen\Desktop\phd\dataset\training\05_transcript_coordinate_mapping\SOD1_refseq_transcript_fetched_from_NCBI.fasta


In [17]:
df.to_csv(STEP7D_OUT, index=False)

print("Saved:")
print(STEP7D_OUT)

df[
    [
        "ISIS",
        "Sequence",
        "n_transcript_hits",
        "target_start_1based",
        "target_end_1based_inclusive",
        "target_relative_midpoint",
        "target_has_exact_refseq_match",
        "target_mapping_unique",
    ]
].head()

Saved:
C:\Users\oozgen\Desktop\phd\dataset\training\06_biophysical_features\SOD1_process_v4d_transcript_coordinate_features.csv


,ISIS,Sequence,n_transcript_hits,target_start_1based,target_end_1based_inclusive,target_relative_midpoint,target_has_exact_refseq_match,target_mapping_unique
0,150478,ATTGAAACAGACATTTTAAC,1,740.0,759.0,0.837430,True,True
1,150494,TCATAATAAGTGCCATACAG,1,845.0,864.0,0.954749,True,True
2,489531,AAATCAGTTTCTCACTACAG,1,681.0,700.0,0.771508,True,True
3,590442,CGCTGCAGGAGACTACG,1,5.0,21.0,0.014525,True,True
4,590455,CTTTCCTTCTGCTCGAA,1,138.0,154.0,0.163128,True,True


In [19]:
# ============================================================
# 7F. Basic DNA Tm proxy
# ============================================================

df = pd.read_csv(STEP7D_OUT)
df.columns = df.columns.str.strip()

# Defragment after many previous column additions, especially gap_3mer_* columns
df = df.copy()

tm_cols = [
    "basic_DNA_Tm_NN_full_sequence",
    "basic_DNA_Tm_NN_gap_sequence",
]

df = df.drop(columns=[c for c in tm_cols if c in df.columns], errors="ignore")

try:
    from Bio.SeqUtils import MeltingTemp as mt

    def basic_dna_tm_nn(seq):
        seq = str(seq).upper()

        if len(seq) < 2:
            return np.nan

        try:
            return mt.Tm_NN(seq, nn_table=mt.DNA_NN4)
        except Exception:
            return np.nan

    df["basic_DNA_Tm_NN_full_sequence"] = df["Sequence"].apply(basic_dna_tm_nn)
    df["basic_DNA_Tm_NN_gap_sequence"] = df["gap_sequence"].apply(basic_dna_tm_nn)

    print("Added basic DNA Tm features.")

except ImportError:
    df["basic_DNA_Tm_NN_full_sequence"] = np.nan
    df["basic_DNA_Tm_NN_gap_sequence"] = np.nan

    print("Biopython is not installed. Tm features set to NaN.")

df.to_csv(STEP7E_OUT, index=False)

print("Saved:")
print(STEP7E_OUT)

df[
    [
        "ISIS",
        "Sequence",
        "gap_sequence",
        "basic_DNA_Tm_NN_full_sequence",
        "basic_DNA_Tm_NN_gap_sequence",
    ]
].head()

Added basic DNA Tm features.
Saved:
C:\Users\oozgen\Desktop\phd\dataset\training\06_biophysical_features\SOD1_process_v4e_basic_tm_features.csv


,ISIS,Sequence,gap_sequence,basic_DNA_Tm_NN_full_sequence,basic_DNA_Tm_NN_gap_sequence
0,150478,ATTGAAACAGACATTTTAAC,AACAGACATT,41.725012,15.270056
1,150494,TCATAATAAGTGCCATACAG,ATAAGTGCCA,44.144247,19.057328
2,489531,AAATCAGTTTCTCACTACAG,AGTTTCTCAC,44.534057,17.263009
3,590442,CGCTGCAGGAGACTACG,CAGGAGA,51.585743,-2.884615
4,590455,CTTTCCTTCTGCTCGAA,CTTCTGC,45.280916,0.220283


In [20]:
# ============================================================
# 7G. RDKit molecular descriptors from SMILES
# ============================================================

df = pd.read_csv(STEP7E_OUT)
df.columns = df.columns.str.strip()

# Defragment after many previous column additions, especially gap_3mer_* columns
df = df.copy()

rdkit_cols = [
    "canonical_smiles",
    "rdkit_valid_smiles",
    "rdkit_mol_wt",
    "rdkit_heavy_atom_count",
    "rdkit_num_atoms",
    "rdkit_num_bonds",
    "rdkit_num_rings",
    "rdkit_tpsa",
    "rdkit_num_h_donors",
    "rdkit_num_h_acceptors",
    "rdkit_formal_charge",
]

df = df.drop(columns=[c for c in rdkit_cols if c in df.columns], errors="ignore")

if "Smiles" not in df.columns:
    print("No Smiles column found. RDKit descriptors skipped.")

    for col in rdkit_cols:
        df[col] = pd.NA

else:
    try:
        from rdkit import Chem
        from rdkit.Chem import Descriptors, rdMolDescriptors

        def mol_from_smiles(smiles):
            try:
                return Chem.MolFromSmiles(str(smiles))
            except Exception:
                return None

        def canonical_smiles(smiles):
            mol = mol_from_smiles(smiles)

            if mol is None:
                return pd.NA

            return Chem.MolToSmiles(mol, canonical=True)

        def rdkit_descriptors(smiles):
            mol = mol_from_smiles(smiles)

            if mol is None:
                return {
                    "rdkit_valid_smiles": False,
                    "rdkit_mol_wt": np.nan,
                    "rdkit_heavy_atom_count": np.nan,
                    "rdkit_num_atoms": np.nan,
                    "rdkit_num_bonds": np.nan,
                    "rdkit_num_rings": np.nan,
                    "rdkit_tpsa": np.nan,
                    "rdkit_num_h_donors": np.nan,
                    "rdkit_num_h_acceptors": np.nan,
                    "rdkit_formal_charge": np.nan,
                }

            return {
                "rdkit_valid_smiles": True,
                "rdkit_mol_wt": Descriptors.MolWt(mol),
                "rdkit_heavy_atom_count": Descriptors.HeavyAtomCount(mol),
                "rdkit_num_atoms": mol.GetNumAtoms(),
                "rdkit_num_bonds": mol.GetNumBonds(),
                "rdkit_num_rings": rdMolDescriptors.CalcNumRings(mol),
                "rdkit_tpsa": rdMolDescriptors.CalcTPSA(mol),
                "rdkit_num_h_donors": rdMolDescriptors.CalcNumHBD(mol),
                "rdkit_num_h_acceptors": rdMolDescriptors.CalcNumHBA(mol),
                "rdkit_formal_charge": Chem.GetFormalCharge(mol),
            }

        df["canonical_smiles"] = df["Smiles"].apply(canonical_smiles)

        rdkit_feature_df = pd.DataFrame(
            df["Smiles"].apply(rdkit_descriptors).tolist()
        )

        df = pd.concat(
            [df.reset_index(drop=True), rdkit_feature_df.reset_index(drop=True)],
            axis=1,
        )

        print("RDKit descriptor status:")
        print(df["rdkit_valid_smiles"].value_counts(dropna=False))

    except ImportError:
        print("RDKit is not installed. RDKit descriptors set to NA.")

        for col in rdkit_cols:
            df[col] = pd.NA

df.to_csv(STEP7F_OUT, index=False)

print("Saved:")
print(STEP7F_OUT)

df[
    [
        "ISIS",
        "Smiles",
        "canonical_smiles",
        "rdkit_valid_smiles",
        "rdkit_mol_wt",
        "rdkit_heavy_atom_count",
        "rdkit_tpsa",
    ]
].head()

RDKit descriptor status:
rdkit_valid_smiles
True    1026
Name: count, dtype: int64
Saved:
C:\Users\oozgen\Desktop\phd\dataset\training\06_biophysical_features\SOD1_process_v4f_rdkit_features.csv


,ISIS,Smiles,canonical_smiles,rdkit_valid_smiles,rdkit_mol_wt,rdkit_heavy_atom_count,rdkit_tpsa
0,150478,COCCO[C@@H]1[C@H](O)[C@@H](C[O]P(=O)([S-])O[C@...,COCCO[C@@H]1[C@H](O)[C@@H](COP(=O)([S-])O[C@H]...,True,7134.996,457,2402.38
1,150494,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,COCCO[C@@H]1[C@H](OP(=O)([S-])OC[C@H]2O[C@@H](...,True,7135.984,457,2428.40
2,489531,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,COCCO[C@@H]1[C@H](OP(=O)([S-])OC[C@H]2O[C@@H](...,True,7086.944,453,2384.96
3,590442,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,COCCO[C@@H]1[C@H](OP(=O)([S-])OC[C@H]2O[C@@H](...,True,6074.022,390,2143.82
4,590455,COCCO[C@@H]1[C@H](OP(=O)([S-])[O]C[C@H]2O[C@@H...,COCCO[C@@H]1[C@H](OP(=O)([S-])OC[C@H]2O[C@@H](...,True,5940.925,379,1981.43


In [21]:
# 8. Final validation and model-ready export

In [22]:
# ============================================================
# 8A. Load final Step 7 file
# ============================================================

df = pd.read_csv(STEP7F_OUT)
df.columns = df.columns.str.strip()

print("Loaded:", STEP7F_OUT)
print("Shape:", df.shape)

Loaded: C:\Users\oozgen\Desktop\phd\dataset\training\06_biophysical_features\SOD1_process_v4f_rdkit_features.csv
Shape: (1026, 162)


In [23]:
# ============================================================
# 8B. Final validation checks
# ============================================================

checks = {}

# Core preprocessing checks
for col in [
    "lengths_match",
    "valid_sequence",
    "valid_chemical_pattern",
    "linkage_count_valid",
    "chem_count_sum_valid",
    "base_count_sum_valid",
    "gap_length_valid",
    "gap_3mer_count_valid",
]:
    if col in df.columns:
        checks[f"{col}_all_true"] = bool(df[col].all())

# Inhibition grouping check, if present
if "inhibition_group_4_equal_count" in df.columns:
    checks["inhibition_groups_present"] = df["inhibition_group_4_equal_count"].notna().all()

# 2-way sequence leakage check, if present
if "split_2way" in df.columns:
    train_sequences = set(df.loc[df["split_2way"] == "train", "Sequence"])
    test_sequences = set(df.loc[df["split_2way"] == "test", "Sequence"])

    checks["no_sequence_leakage_train_test_2way"] = (
        len(train_sequences.intersection(test_sequences)) == 0
    )

# RDKit validity check, if present
if "rdkit_valid_smiles" in df.columns:
    if df["rdkit_valid_smiles"].notna().any():
        checks["all_rdkit_smiles_valid_or_na"] = bool(
            df["rdkit_valid_smiles"].dropna().astype(bool).all()
        )

# Transcript mapping check, if present
if "target_has_exact_refseq_match" in df.columns:
    checks["transcript_mapping_column_present"] = True
    checks["at_least_one_transcript_match"] = bool(df["target_has_exact_refseq_match"].sum() > 0)

validation_summary = pd.DataFrame({
    "check": list(checks.keys()),
    "passed": list(checks.values()),
})

display(validation_summary)

if not validation_summary["passed"].all():
    print("One or more validation checks failed. Review validation_summary before modeling.")
else:
    print("All final validation checks passed.")

,check,passed
0,lengths_match_all_true,True
1,valid_sequence_all_true,True
2,valid_chemical_pattern_all_true,True
3,linkage_count_valid_all_true,True
4,chem_count_sum_valid_all_true,True
5,base_count_sum_valid_all_true,True
6,gap_length_valid_all_true,True
7,gap_3mer_count_valid_all_true,True
8,inhibition_groups_present,True
9,no_sequence_leakage_train_test_2way,True


All final validation checks passed.


In [24]:
# ============================================================
# 8C. Save model-ready dataset
# ============================================================

df.to_csv(FINAL_OUT, index=False)

data_dictionary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "n_missing": [df[col].isna().sum() for col in df.columns],
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
})

data_dictionary.to_csv(FINAL_DATA_DICTIONARY_OUT, index=False)
validation_summary.to_csv(FINAL_VALIDATION_OUT, index=False)

print("Saved:")
print(FINAL_OUT)
print(FINAL_DATA_DICTIONARY_OUT)
print(FINAL_VALIDATION_OUT)

Saved:
C:\Users\oozgen\Desktop\phd\dataset\training\07_model_ready_dataset\SOD1_model_ready_v1.csv
C:\Users\oozgen\Desktop\phd\dataset\training\07_model_ready_dataset\SOD1_model_ready_v1_data_dictionary.csv
C:\Users\oozgen\Desktop\phd\dataset\training\07_model_ready_dataset\SOD1_model_ready_v1_validation_summary.csv
